In [4]:
import regex as re
import pickle
import ast
from typing import Iterable, Iterator


class Tokenizer:
    def __init__(
        self, vocab: dict[int, bytes], 
        merges: list[tuple[bytes, bytes]], 
        special_tokens: list[str] | None = None
        ):
        '''从给定的词表、合并规则和（可选的）特殊 tokens 构造一个分词器'''
        self.vocab = vocab
        self.vocab_inverse = {v: k for k, v in vocab.items()}
        self.merges = merges
        self.merges_ranked = {k: v for v, k in enumerate(merges)}
        self.special_tokens = special_tokens
    
    @classmethod
    def from_files(cls, vocab_filepath: str, 
                   merges_filepath: str, 
                   special_tokens: list[str] | None = None):
        '''从序列化的 vocab 和 merges 文件构造一个 Tokenizer'''
        
        with open(vocab_filepath, 'rb') as f:
            vocab = pickle.load(f)
        with open(merges_filepath, 'r') as f:
            merges = [ast.literal_eval(line.strip()) for line in f]
            
        return cls(vocab, merges, special_tokens)
    
    def encode(self, text: str) -> list[int]:
        '''将输入文本编码为 token ID 序列'''
        PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
        pre_tokens = re.findall(PAT, text)

        idx = []
        for pre_token in pre_tokens:
            pre_token = [i.encode('utf-8') for i in pre_token]
            while True:
                new_pre_token = []
                to_merge = dict()
                for index1, index2 in zip(pre_token, pre_token[1:]):
                    pair = (index1, index2)
                    if pair in self.merges_ranked:
                        to_merge[pair] = self.merges_ranked[pair]
                if len(to_merge) == 0:
                    break
                # 找到合并优先级最高的
                best_pair = min(to_merge, key=to_merge.get)
                # 合并
                i = 0
                while i < len(pre_token):
                    if i + 1 < len(pre_token) and pre_token[i] == best_pair[0] and pre_token[i + 1] == best_pair[1]:
                        new_pre_token.append(best_pair[0] + best_pair[1])
                        i += 2
                    else:
                        new_pre_token.append(pre_token[i])
                        i += 1
                pre_token = new_pre_token.copy()
            for i in pre_token:
                idx.append(i)
        idx = [self.vocab_inverse[i] for i in idx]
        
        return idx
        
    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        '''给定字符串的可迭代对象（如文件句柄），惰性地产生 token IDs'''
        for text in iterable:
            for tid in self.encode(text):
                yield tid
        
    def decode(self, ids: list[int]) -> str:
        '''将 token ID 序列解码回原始文本'''
        byte_sequence = b"".join(self.vocab[i] if i in self.vocab else bytes([i]) for i in ids)
        # 使用 errors="replace" 保证非法字节能被替换成 �
        return byte_sequence.decode("utf-8", errors="replace")

In [5]:
special_tokens = ["<|endoftext|>"]
ceshi = Tokenizer.from_files('result/TinyStories_vocab.pkl', 
                             'result/TinyStories_merges.txt', special_tokens)

In [175]:
ceshi.encode('i loved you!')

[105, 502, 346, 33]

In [176]:
ceshi.decode([105, 502, 346, 33])

'i loved you!'

In [6]:
from test1_bpe_optimize1 import run_train_bpe
import regex as re

vocab, merges = run_train_bpe('../data/text_example2.txt', 260, special_tokens)
vocab_inverse = {v: k for k, v in vocab.items()}
merges_ranked = {k: v for v, k in enumerate(merges)}

In [10]:

with open('../data/text_example2.txt', 'r') as f:
    text = example = f.read()
    
text = '你好'
    
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
pre_tokens = re.findall(PAT, text)

idx = []
for pre_token in pre_tokens:
    pre_token = [i.encode('utf-8') for i in pre_token]
    while True:
        new_pre_token = []
        to_merge = dict()
        for index1, index2 in zip(pre_token, pre_token[1:]):
            pair = (index1, index2)
            if pair in merges_ranked:
                to_merge[pair] = merges_ranked[pair]
        if len(to_merge) == 0:
            break
        # 找到合并优先级最高的
        best_pair = min(to_merge, key=to_merge.get)
        # 合并
        i = 0
        while i < len(pre_token):
            if i + 1 < len(pre_token) and pre_token[i] == best_pair[0] and pre_token[i + 1] == best_pair[1]:
                new_pre_token.append(best_pair[0] + best_pair[1])
                i += 2
            else:
                new_pre_token.append(pre_token[i])
                i += 1
        pre_token = new_pre_token.copy()
    for i in pre_token:
        idx.append(i)
        
idx2 = []
for i in idx:
    if i in vocab_inverse:
        idx2.append(vocab_inverse[i])
    else:
        idx2.append(i)
idx2

[b'\xe4\xbd\xa0', b'\xe5\xa5\xbd']

In [8]:
idx

[108,
 259,
 44,
 32,
 108,
 259,
 44,
 32,
 108,
 259,
 44,
 32,
 108,
 259,
 44,
 32,
 108,
 259,
 44,
 10,
 108,
 259,
 101,
 114,
 44,
 32,
 108,
 259,
 101,
 114,
 44,
 10,
 119,
 105,
 100,
 258,
 44,
 32,
 119,
 105,
 100,
 258,
 44,
 32,
 119,
 105,
 100,
 258,
 44,
 10,
 110,
 101,
 119,
 258,
 44,
 32,
 110,
 101,
 119,
 258,
 44,
 32,
 110,
 101,
 119,
 258,
 44,
 32,
 110,
 101,
 119,
 258,
 44,
 32,
 110,
 101,
 119,
 258,
 44,
 32,
 110,
 101,
 119,
 258]